In [1]:
files = {
	# ===== DeBERTa 512 =====
	"deberta_42":     "deberta_processed/deberta_seed_42_processed/submission_seed42.csv",
	"deberta_1337":   "deberta_processed/deberta_seed_1337_processed/submission_seed_daberta_processed1337.csv",
	"deberta_2024":   "deberta_processed/deberta_seed_2024_processed/submission_seed_daberta_processed2024.csv",
	"deberta_noseed": "deberta_processed/submission_deberta_processed_noseed_MAXLEN_512.csv",

	# ===== DeBERTa 256 =====
	"deberta256_noseed": "deberta_MAXLEN256/submission_deberta_processed_noseed_MAXLEN_256.csv",
    "deberta256_42" :  "deberta_MAXLEN256/deberta_seed42_MAXLEN256\submission_roberta_processed_seed42_MAXLEN256.csv",
    "deberta256_1337" : "deberta_MAXLEN256/deberta_seed1337_MAXLEN256/submission_roberta_processed_seed1337_MAXLEN256.csv",
    "deberta256_2024" : "deberta_MAXLEN256/deberta_seed2024_MAXLEN256/submission_roberta_processed_seed2024_MAXLEN256.csv",

	# ===== RoBERTa 512 =====
	"roberta_42":     "roberta_processed/roberta_processed_seed_42/submission_seed_roberta_processed42.csv",
	"roberta_1337":   "roberta_processed/roberta_processed_seed_1337/submission_seed_roberta_processed1337.csv",
	"roberta_2024":   "roberta_processed/roberta_processed_seed_2024/submission_seed_roberta_processed2024.csv",
	"roberta_noseed": "roberta_processed/submission_roberta_processed_noseed_MAXLEN_512.csv",

	# ===== RoBERTa 256 =====
	"roberta256_42":   "roberta_MAXLEN256/roberta_seed42_MAXLEN256/submission_roberta_processed_seed42_MAXLEN256.csv",
	"roberta256_1337": "roberta_MAXLEN256/roberta_seed1337_MAXLEN256/submission_roberta_processed_seed1337_MAXLEN256.csv",
	"roberta256_2024": "roberta_MAXLEN256/roberta_seed2024_MAXLEN256/submission_roberta_processed_seed2024_MAXLEN256.csv",
    "roberta_256_noseed" : "roberta_MAXLEN256/submission_roberta_processed_noseed_MAXLEN_256.csv"
}


In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict

# carichiamo tutte le submission
preds = {}

for name, path in files.items():
	df = pd.read_csv(path)
	df = df.sort_values("Id").reset_index(drop=True)
	preds[name] = df["Predicted"].values

# sanity check
ids = pd.read_csv(next(iter(files.values())))["Id"].values
N = len(ids)
print("Samples:", N)


Samples: 20000


In [3]:
# matrice: (n_samples, n_models)
pred_matrix = np.vstack(list(preds.values())).T

# numero di label distinte per sample
n_unique = np.array([len(set(row)) for row in pred_matrix])

# hard cases (base)
hard_mask = n_unique > 1
hard_indices = np.where(hard_mask)[0]

print(f"Hard cases (>=2 models disagree): {len(hard_indices)} "
      f"({len(hard_indices)/N:.2%})")


Hard cases (>=2 models disagree): 6839 (34.20%)


In [ ]:
# separate for architecture
roberta_models = [k for k in preds if "roberta" in k.lower()]
deberta_models = [k for k in preds if "deberta" in k.lower()]

roberta_preds = np.vstack([preds[k] for k in roberta_models]).T
deberta_preds = np.vstack([preds[k] for k in deberta_models]).T



In [5]:
def cross_arch_disagreement(r_row, d_row):
	return len(set(r_row).union(set(d_row))) > 1 and \
	       len(set(r_row)) == 1 and len(set(d_row)) == 1 and \
	       r_row[0] != d_row[0]

strong_hard_mask = np.array([
	cross_arch_disagreement(roberta_preds[i], deberta_preds[i])
	for i in range(N)
])

strong_hard_indices = np.where(strong_hard_mask)[0]

print(f"Strong hard cases (RoBERTa vs DeBERTa): {len(strong_hard_indices)} "
      f"({len(strong_hard_indices)/N:.2%})")


Strong hard cases (RoBERTa vs DeBERTa): 11 (0.06%)


In [6]:
def vote_entropy(row):
	labels, counts = np.unique(row, return_counts=True)
	p = counts / counts.sum()
	return -np.sum(p * np.log(p))

entropies = np.array([vote_entropy(row) for row in pred_matrix])

# high-ambiguity hard cases (es. entropia sopra soglia)
entropy_thr = np.percentile(entropies[hard_mask], 50)
very_hard_mask = (entropies >= entropy_thr) & hard_mask

print(f"High-entropy hard cases: {very_hard_mask.sum()}")


High-entropy hard cases: 3563


In [7]:
hardcase_df = pd.DataFrame({
	"Id": ids,
	"is_hard": hard_mask,
	"is_strong_hard": strong_hard_mask,
	"vote_entropy": entropies
})

hardcase_df.to_csv("hard_cases_analysis.csv", index=False)


In [9]:
x = np.load("roberta_MAXLEN256/logits_roberta_processed_noseed_MAXLEN_256.npy")

In [10]:
print(x.shape)

(20000, 7)
